# **Dispatch / Combine 核心流程拆解（non-quant）**

本节从算法视角拆解 `Dispatch` 与 `Combine` 在 AIV 核上执行时的核心阶段：每一步做什么、依赖什么、产出什么。本节聚焦流程与数据流动，暂不展开具体接口签名。

**学习目标**：理解 Dispatch 的四个核心阶段与 Combine 的两个核心阶段在 Win 区与 HBM 上的数据流动顺序与同步关系。

**术语对齐**（与代码保持一致，后续各小节通用）：

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">文档表述</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">对应代码标识</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">含义</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AIV 核</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>aivId_</code> / <code>aivNum_</code> / <code>aivUsedAllToAll_</code> / <code>aivUsedCumSum_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">矢量核</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本卡 <code>epRankId</code> / 全局 <code>epWorldSize</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>epRankId_</code> / <code>epWorldSize_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">源卡编号与 EP 域大小</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">全局 / 本地专家数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>moeExpertNum_</code> / <code>moeExpertNumPerRank_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家在 EP 域上的分布</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">路由结果 <code>expertIds</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertIdsGM_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">形状 <code>[bs, k]</code></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">三元组 <code>(epRankId, tokenIndex, topkIdx)</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>FillTriple</code> 拼装</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Combine 回送数据时定位用</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Win数据区 / Win状态区</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>selfRankWinIn</code> 数据部分 / <code>GetSymMemStatusRegionAddr()</code> 状态部分</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">同一块共享内存的两类用途</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">状态块</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>statusTensor_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">一个 <code>UB_ALIGN</code> 对齐块（32 字节，8 个 int32），首位放 flag、第二位放 count</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">接收计数 <code>epRecvCnt</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>sendCntTensor_</code> / <code>sendCountsOutGM_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各源卡发来的 token 数 / 累积偏移</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">本地专家 token 数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expertTokenNumsOutGMTensor_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">由 <code>expertTokenNumsType</code> 决定是否累积</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">整理后特征 <code>expandX</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expandXOutGM_</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Dispatch 最终的 HBM 输出</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">双 buffer</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>BUFFER_NUM = 2</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">搬运 / 远端写的流水缓冲</td>
  </tr>
</table>

---

## **1. Dispatch 核心流程（4 步）**

Dispatch 把本卡 batch 中的每个 token 按 TopK 路由结果分发到对应专家所在卡，并同步产出供 Combine 与下游 FFN 使用的辅助信息。算法层分四步：**发送数据 → 发送 flag 与 count → 等待并计算偏移 → 本地整理**。

实现上第 1 步与第 2 步由不同 AIV 核分组并行执行（`aivId_ < aivUsedAllToAll_` 发数据，其余核算并发送计数）；1.3 复用 1.2 那组核；1.4 由全部核共同完成。本节按算法顺序说明，并在 1.5 节说明并行关系的汇合。

<img src="./images/02.06_dispatch_core_processes.png" width="300">

---

### **1.1 循环发送数据**

**目的**：把本卡 `x` 中的每个 token 按 `expertIds` 列出的 K 个目标专家分别拷贝一份，写入这些专家所在卡的 Win数据区中预先约定的位置。

**核心数据流**：

1. 把 `expertIds`（形状 `[bs, k]`）整体搬入 UB，按「当前迭代的目标专家」生成掩码，仅挑出命中该专家的 (token, topk) 对——保证发往同一专家的 token 在对端 Win数据区上连续排列。
2. 对每个被选中的 (token, topk) 对：
    - HBM → UB：把这条 token 的特征向量（`h` 个元素）搬到 UB；
    - 拼三元组：在该 token 末尾追加 `(epRankId, tokenIndex, topkIdx)`，作为 Combine 回送数据时的定位依据；
    - UB → 对端 Win数据区：通过共享内存写指令把「token 主体 + 三元组」整体推到目标卡上对应专家的槽位（图 A）。
3. 以上三个动作通过双 buffer（`BUFFER_NUM = 2`）流水并行，使搬运与远端写在不同 buffer 上同时推进。

**图 A · 一条 token 的传输路径**：

<img src="./images/02.06_token_transmission_path.png" width="1500">

**输入 / 输出**：

- 输入：`x`、`expertIds`。
- 输出：分散写入到所有目标卡的 Win数据区；本卡 HBM 上此时尚未产出最终输出。

**实现说明**：

- 「写到对端 Win数据区」不等价于「Combine 输入已就绪」：对端仅完成数据接收，尚未获得 token 到达数量信息，计数与同步由 1.2 完成。
- 三元组写在 token 尾部而非单独表，是为了对端整理时一次连续 DMA 同时取得「特征 + 来源信息」。
- 本步骤只发数据、不统计数量；统计由另一组 AIV 核并行完成。

---

### **1.2 发送 flag 与 count**

**目的**：把「本卡为每个目标专家准备了多少条 token」连同一个就绪 flag 写到目标卡 Win状态区对应槽位，为对端提供唯一可信的就绪信号与数量来源。

**核心数据流**：

1. **AIV 核分组与专家切片**：本步骤由 `aivId_ >= aivUsedAllToAll_` 的另一组核执行，与 1.1 并行。该组核按 `moeExpertNum` 均匀切片，每核负责若干专家的计数发送。
2. **逐专家计数**：对每个负责的目标专家，在 UB 上对 `expertIds` 中命中该专家的位置求和，得到「本卡将发往该专家的 token 数」。该统计完全在 UB 上完成，不重复访问 HBM。
3. **组装并整体推送**：在 UB 上把 flag 与 count 装进同一个 UB 对齐块（图 B），再用一次共享内存写整体推到目标卡 Win状态区的固定槽位——槽位地址由「目标专家本地编号」与源 `epRankId` 共同决定。

**图 B · AIV 核分组专家切片与逐专家计数**：

<img src="./images/02.06_AIV_core_dispatching_and_per_expert_counting.png" width="1500">

**图 C · 状态块布局与写入**：

<img src="./images/02.06_state_block_layout_and_writing.png" width="1500">

**输入 / 输出**：

- 输入：`expertIds`。
- 输出：分散写入到所有目标卡的 Win状态区；每条为 flag + count 固定大小组合。

**实现说明**：

- flag 与 count 装在同一对齐块、由同一次共享内存写整体推送，因此对端读到非零 flag 时 count 必然已就位，无需额外协调。
- 本步骤统计的是「将要发」而非「已经发完」；1.1 与 1.2 在源卡上无先后关系，仅靠对端 Win 区上「数据区 / 状态区」位置区分。

---

### **1.3 等待 flag 与计算接收偏移**

**目的**：等所有源卡都把要发给本卡的 count 写完，然后按本地专家维度算出每个专家在 `expandX` 上的起始偏移（即 `epRecvCnt`），供 1.4 直接消费。

**核心数据流**（参见图 D）：

1. **切片本卡接收槽**：复用 1.2 的核组，按本卡接收状态总条数（`moeExpertNumPerRank × epWorldSize`）重新均匀切片，每核负责一段连续状态槽。
2. **卡间同步**：每核轮询自己段的 Win状态区，对 flag 求和直到达到「负责槽数」目标值；flag 到齐后立即清状态区，避免遗留状态污染下一轮。
3. **核间软同步 + 前缀和**：每核把段内 count 求和写到本卡 Win 区一段约定的「核间汇总区」，相互轮询直到 `aivUsedCumSum_` 个核全部就绪；再以「前面所有核的汇总」作为本核第一个本地专家的起始偏移，核内逐专家累加得到完整偏移序列。
4. **结果对外可见**：写 `sendCountsOut`（即 `epRecvCnt`）+ 复制 `aivNum` 份到工作空间 + 向 Win 区另一段写「本地整理同步 flag」通知 1.4。

**图 D · 两层同步与前缀和**：

<img src="./images/02.06_two-layer_synchronization_and_prefix_sum.png" width="600">

**输入 / 输出**：

- 输入：本卡 Win状态区上由所有源卡写入的 flag 与 count。
- 输出：`epRecvCnt`（HBM 接收偏移）、工作空间偏移副本、本地整理同步 flag。

**实现说明**：

- 两层同步：卡间同步基于「对端写 flag、本卡求和」，核间软同步基于「核间汇总区轮询」，二者都不依赖额外硬件原语。
- 前缀和粒度是「本地专家」而非「单 token」；最终 `expertTokenNums` 是「逐专家计数」还是「累积」由 `expertTokenNumsType` 控制。

---

### **1.4 本地整理（LocalWindowCopy）**

**目的**：把本卡 Win数据区上按 (源卡, 本地专家) 分散排列的 token，按本地专家维度连续化整理后写到 HBM 上的 `expandX`，三元组同步落到 `expandIdx`（图 E）。

**核心数据流**：

1. **跨阶段同步**：全部 AIV 核共同参与；进入前先轮询 1.3 写下的「本地整理同步 flag」，确认 `epRecvCnt` 与工作空间副本已就绪。
2. **按 (源卡, 本地专家) 槽位分核**：本卡共有 `moeExpertNumPerRank × epWorldSize` 个槽位，均匀切给 `aivNum` 个核；每核据此构造一张「自己负责的有效 (源卡, 专家) 对列表」。
3. **token 粒度到达确认**：源卡写一条 token 时会同时置位一个 token 级 flag。本步骤搬运每批 token 前先比对该 flag，未到齐则跳过、轮询下一个 (源卡, 专家) 对，避免单点等待。token-flag 的具体结构留到下一节「Win 内存布局」展开。
4. **批量搬运 + 写出**：已到齐的 token 一次最多搬 `maxCopyTokenCnt` 个，从 Win数据区入 UB；目标行号由「源卡前缀和（来自 `sendCntTensor_`）」+「本核针对该对此前已搬数」相加得到。token 主体写 `expandX`，三元组写 `expandIdx`。
5. **清 token-flag**：一个 (源卡, 专家) 对的全部 token 搬完后立即清掉对应 token-flag，为下一轮 Dispatch 准备。
6. **`expertTokenNums` 输出**：由 1.3 中 `aivId_ == aivUsedAllToAll_` 的核额外完成，按本地专家维度求和后写 `expertTokenNumsOut`，是否累积由 `expertTokenNumsType` 控制。

**图 E · 整理前后布局**：

![整理前后布局](./images/02.06_organize_the_layout_before_and_after.png)

**输入 / 输出**：

- 输入：本卡 Win数据区上的 token 与 token-flag、1.3 写出的偏移副本与本地整理同步 flag。
- 输出：HBM 上的 `expandX` / `expandIdx`；Win数据区上的 token-flag 被清零。

**实现说明**：

- 1.3 的「专家粒度 count flag」与 1.4 的「token 粒度 flag」是两个层级的标记位：前者保证「该源卡的整段已发完」，后者保证「Win数据区上具体某条 token 已写完」。
- 目标行号是「源卡前缀和 + 核内已搬数」两部分相加，前者保证不同源卡的输出段不重叠，后者保证多次批量搬运能续上。
- 1.4 结束后 Dispatch 的全部对外输出才就绪，下游 FFN 在此之前不能读输出。

---

### **1.5 关键设计点小结**

- **核分组并行**：1.1 发数据 / 1.2 算并发送计数 由两组 AIV 核同时跑，1.3 复用 1.2 组，1.4 全员参与。
- **同步全部走 Win 区**：源卡↔本卡、本卡核间，全部依赖「写 flag + 轮询求和」，无额外硬件原语。
- **位置即协议**：所有目的地址都可由 `epRankId` / 本地专家编号 / 已发或已收数推算，不交换协商信息。
- **HBM 输出就绪时序**：`expertTokenNums` 在 1.3 由 0 号 cumsum 核写；`epRecvCnt` 在 1.3 末段写；`expandX` / `expandIdx` 直到 1.4 才写完。下游消费以「1.4 全员执行完」为安全等待点。

---

## **2. Combine 核心流程（2 步）**

Combine 把 dispatch 阶段已经分发并经 FFN（或上游算子）处理过的 token 按 TopK 路由结果回送到各原源卡，并在原源卡上对每条 token 的 K 路 expert 输出做加权求和，得到最终输出 `XOut`。算法层可以划分为两步：**回送数据并写 token-flag → 等待并加权求和**。

实现上 Combine 阶段所有 AIV 核共同参与，但两步使用 **不同的分核维度**：2.1 按「本卡 dispatch 阶段已接收的 token 总数」分核，2.2 按「本卡原始 batch」分核。Combine 不再像 dispatch 那样区分两组核，而是先做完 2.1，再统一进入 2.2。

<img src="./images/02.06_combine_core_processes.png" width="300">

---

### **2.1 回送数据并写 token-flag（SetWaitStatusAndDisPatch）**

**目的**：把本卡在 dispatch 阶段已经接收并整理好的 `expandX` 中的每条 token，按 dispatch 写下的三元组 `(epRankId, tokenIndex, topkIdx)`，逐条回送给 **原源卡** 的 Win数据区，并写一个 token-flag 到原源卡 Win状态区，作为「该 token 的某个 topk 拷贝已经回到」的就绪信号。

**核心数据流**（参见图 F）：

1. **AIV 核分核**：以本卡 dispatch 阶段已接收的 token 总数 `selfSendCnt_`（从 `epSendCount` 末项读取）为基准，在 `aivNum_` 个核之间均匀切片，每核负责 `[startTokenId_, endTokenId_)` 一段连续接收 token。`coreIdx_ >= selfSendCnt_` 的核空转。
2. **读三元组**：把这段 token 对应的 `expandIdx[startTokenId_*3 : endTokenId_*3]`（每条 3 个 uint32）整段搬入 UB。
3. **逐 token 回送**——循环 `loop = 0..sendCntNum_-1`，并以 `(loop + epRankId_) % sendCntNum_` 做错峰索引（不同源卡上的核错开访问目的卡，缓解写冲突）：
    - 从三元组中拆出目的卡 `toRankId`、源卡上原始 token 下标 `tokenId`、topk 下标 `topkId`；
    - 从 HBM `expandX[tkIndex * axisH_]` 把这条 token 主体搬入 UB；
    - 通过共享内存写指令把 UB 中的 token 推到 **目的卡 Win数据区** 上 `(tokenId * K + topkId) * hAlignWinSize_` 槽位——按「目的 token × topk」连续布局，与 dispatch 阶段「源卡 × 本地专家」连续布局形成对偶。
4. **紧随写 token-flag**：`PipeBarrier<PIPE_MTE3>` 之后，把一个 UB 对齐块（首位为 1.0、其余为 0）共享内存写到 **目的卡 Win状态区** 上 `tokenId * flagRcvCount_ * stateOffset_ + topkId * stateOffset_` 槽位——按 `(目的 tokenId, topkId)` 索引；一条目的 token 共 `K` 个 flag 槽，K 路都到齐则求和等于 `K`。

**图 F · 一条接收 token 回送到原源卡的路径**：

<img src="./images/02.06_token_return.png" width="1200">

**输入 / 输出**：

- 输入：本卡 HBM 上的 `expandX` 与 `expandIdx`（均由 dispatch 阶段写出）。
- 输出：分散写入到所有原源卡的 Win数据区与 Win状态区；本卡 HBM 上此时未产出任何 combine 最终输出。

**实现说明**：

- 本步骤的「目的卡」是 dispatch 阶段的「源卡」，是双向回送的视角切换；同一对 (rank, token, topk) 在 dispatch 与 combine 中扮演相反角色，但三元组完全复用。
- 写 token 与写 token-flag 是两条独立的共享内存写指令，但用 `PipeBarrier<PIPE_MTE3>` 串起来，保证 flag 出现在对端时 token 已落盘——避免对端读到「flag = 1 但 token 还没写完」。
- 错峰索引 `(loop + epRankId_) % sendCntNum_` 是性能优化，不改变正确性：所有源卡仍会逐一回送同样的 token 集合，只是顺序错开。

---

### **2.2 等待 + K 路加权求和（LocalWindowCopy）**

**目的**：对本卡原始 batch 内的每条 token，等所有 K 路 topk 拷贝都回到本卡 Win数据区后，按 `expertScales` 给出的权重对 K 路向量加权求和，最终落到 HBM 的 `XOut[bs, h]`。

**核心数据流**（参见图 G）：

1. **AIV 核分核**：按 `axisBS_`（= 本卡 bs）在全部 `aivNum_` 个核之间均匀切片，每核负责 `[beginIndex, endIndex)` 一段连续 token。
2. **读权重**：将 `expertScales`（形状 `[bs, k]`）整段搬入 UB，供本核所有 token 的 K 路累加复用。
3. **逐 token 处理**——对每条本地 token：
    - **WaitDispatch**：轮询本卡 Win状态区上以 `tokenIndex * flagRcvCount_ * stateOffset_` 起的 `flagRcvCount_ × FLOAT_PER_UB_ALIGN` 个 float（`flagRcvCount_ = axisK_`，每个 topk 一个 UB 对齐块），对其求和等于目标值即代表 K 路 token-flag 全部到齐；到齐后整段清零，准备下一轮。
    - **累加器清零**：UB 上 `sumFloat = 0`，长度 `axisH_`。
    - **K 路读 + 加权累加**：对 `topkId = 0..axisK_-1`，先读 `expertIds[tokenIndex*K + topkId]`，若 ≥ `moeExpertNum_` 视为该路无效，跳过；否则从本卡 Win数据区 `(tokenIndex*K + topkId) * hAlignWinSize_` 起读这条 K 路 token 进 UB，`Cast` 为 float，乘 `expertScales[index]` 后加到累加器。
    - **写出**：累加完毕后 `Cast` 回 X 类型，写到 `XOut[tokenIndex * axisH_]`。

**图 G · K 路加权求和**：

<img src="./images/02.06_K-way_weighted_summation.png" width="600">

**输入 / 输出**：

- 输入：本卡 Win数据区上由所有原目的卡回写的 K 路 token、Win状态区上的 token-flag、HBM 上的 `expertScales` 与 `expertIds`。
- 输出：HBM 上的 `XOut`；Win状态区上的 token-flag 被清零，为下一轮 Combine 准备。

**实现说明**：

- 这里读的 `expertIds` 是 dispatch 阶段使用过的同一份路由结果——combine 与 dispatch 共享这份路由信息，因此能与 dispatch 阶段在「(token, topkId) → 专家」上达成一致。
- 「token-flag 求和 = K」是 Combine 阶段唯一的卡间同步点，与 dispatch 1.3 的「专家粒度求和」属于不同层级。
- 一条本地 token 的 K 路向量分别来自不同卡的 expert 输出，但都被写回到 **本卡 Win数据区** 上以 `(tokenIndex, topkId)` 为索引的相邻槽位，因此本卡 K 路读取在 Win数据区上是局部连续的。

---

### **2.3 关键设计点小结**

- **目的地直接复用三元组**：Combine 不再协商回送目标，dispatch 阶段已把 `(epRankId, tokenIndex, topkIdx)` 写入 `expandIdx`，2.1 顺序读取即可定位目的卡与目的槽位。
- **单层 token-flag 即够用**：dispatch 阶段「专家粒度 count flag」+「token 粒度 token-flag」两层；combine 只保留 token 粒度，每条原始 token 在本卡 Win状态区上占 `K` 个 flag 槽，求和等于 `K` 即代表 K 路全部回到。
- **两个阶段分核维度对偶**：1.1 按「本卡 dispatch 阶段已接收的 token 总数 `selfSendCnt`」分核（每条 token 回送一次），2.2 按「本卡原始 batch `bs`」分核（每条本地 token 累加 K 路）；两种切法分别服务于「我已收的每条 → 还回去」与「我原有的每条 → 聚合回来」两个对偶视角。
- **HBM 输出就绪时序**：`XOut` 直到 2.2 全员执行完才完全可见；2.1 结束仅代表本卡已把所有 token 推到对端 Win，本地 `XOut` 还未产出。下游消费者以「2.2 全员执行完」为安全等待点。

---

**导航**：[← 上一章：Dispatch&Combine输入输出对应关系](02.05_operator_logic_overview.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：Win区布局 →](02.07_win_memory_layout.ipynb)

---